In [ ]:
"""
RT-DETR encoder as Layer 2 classifier — matched to the MobileNetV3 / ViT-S runs.

What this is: RT-DETR-R50vd's backbone + hybrid encoder (ResNet-50-vd -> AIFI
transformer on the /32 map -> CCFM cross-scale fusion), global-pooled at every
scale, then a linear head. The query decoder is dropped: HAM10000 has no boxes,
so there is nothing for it to localise. This IS the RT-DETR encoder; say so.

Same splits.csv, same colour constancy, same weight cap, label smoothing,
two-stage schedule, temperature scaling and escalated-subset check as the
other two runs. Only differences: 512px input (encoder pretrained at 640,
224 throws the point of it away) -> separate image cache, and 20 stage-2
epochs because each epoch is ~3x a ViT-224 epoch.

Colab, GPU. Paste each # %% block into its own cell.
"""

In [1]:

# %% ============================================================ CELL 1: setup
import os, glob, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import kagglehub
DATA = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

!pip install -q "transformers>=4.46" timm
from transformers import RTDetrModel

SEED = 42
OUT = "/content/drive/MyDrive/amsdds"
IMG_SIZE = 512                        # 640 = pretrain res, 512 = 1.5x cheaper
CACHE = f"/content/ham_cache{IMG_SIZE}"   # NOT the 256px cache the other runs use
CACHE_SHORT = int(IMG_SIZE * 1.15)    # shorter side stored, leaves room for crop
BATCH = 16
WEIGHT_CAP = 2.0
USE_COLOR_CONSTANCY = True
MODEL_NAME = "PekingU/rtdetr_r50vd"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")


Mounted at /content/drive
Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
device: cuda Tesla T4


In [2]:

src = {os.path.splitext(os.path.basename(p))[0]: p
       for p in glob.glob(f"{DATA}/**/*.jpg", recursive=True)}
splits_csv = f"{OUT}/splits.csv"
assert os.path.exists(splits_csv), "splits.csv missing -- never re-split"
df = pd.read_csv(splits_csv)
CLASSES = sorted(df.dx.unique())
df["y"] = df.dx.map({c: i for i, c in enumerate(CLASSES)})
df["src"] = df.image_id.map(src)
df["path"] = df.image_id.map(lambda i: f"{CACHE}/{i}.jpg")
assert df.src.isna().sum() == 0
print(df.split.value_counts().to_dict())


AssertionError: splits.csv missing -- never re-split

In [3]:
import os
print(os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive")[:20])
print(os.path.exists("/content/drive/MyDrive/amsdds"))
!ls -la /content/drive/MyDrive/amsdds 2>&1 | head

True
['amsdds']
True
total 8
drwxr-xr-x 2 root root 4096 Sep  3 07:17 .
drwxr-xr-x 3 root root 4096 Sep  3 07:17 ..


In [4]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)   # pick the account that owns amsdds
!ls /content/drive/MyDrive/amsdds

ValueError: Mountpoint must not already contain files

In [5]:
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception as e:
    print("nothing to unmount:", e)
!rm -rf /content/drive
drive.mount("/content/drive")
!ls /content/drive/MyDrive/amsdds

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
baseline_mobilenetv3.pt  layer1_dinov2_v1.pt	     routing_results.json
configs			 layer1_mobilenetv3_best.pt  splits.csv
dinov2_feats.npz	 layer2_train_subset.csv     test_routing_detail.csv
layer1_comparison.csv	 metrics_mobilenetv3.json    threshold_sweep.csv
layer1_dinov2_head.pt	 ood_scores.npz		     threshold_sweep.png


In [6]:

src = {os.path.splitext(os.path.basename(p))[0]: p
       for p in glob.glob(f"{DATA}/**/*.jpg", recursive=True)}
splits_csv = f"{OUT}/splits.csv"
assert os.path.exists(splits_csv), "splits.csv missing -- never re-split"
df = pd.read_csv(splits_csv)
CLASSES = sorted(df.dx.unique())
df["y"] = df.dx.map({c: i for i, c in enumerate(CLASSES)})
df["src"] = df.image_id.map(src)
df["path"] = df.image_id.map(lambda i: f"{CACHE}/{i}.jpg")
assert df.src.isna().sum() == 0
print(df.split.value_counts().to_dict())

{'train': 7116, 'val': 1458, 'test': 1441}


In [7]:

# %% ================================ CELL 3: 512px preprocessed image cache
def shades_of_gray(arr, power=6):
    a = arr.astype(np.float32)
    vec = np.power(np.mean(np.power(a, power), axis=(0, 1)), 1.0 / power)
    vec = vec / (np.sqrt(np.sum(vec ** 2)) + 1e-8)
    return np.clip(a / (vec * np.sqrt(3) + 1e-8), 0, 255).astype(np.uint8)

def build_one(row):
    if os.path.exists(row.path):
        return
    img = Image.open(row.src).convert("RGB")
    w, h = img.size
    s = CACHE_SHORT / min(w, h)                 # HAM is 600x450 -> ~1.3x upsample
    img = img.resize((round(w * s), round(h * s)), Image.BICUBIC)
    if USE_COLOR_CONSTANCY:
        img = Image.fromarray(shades_of_gray(np.array(img)))
    img.save(row.path, quality=95)

need = sum(not os.path.exists(p) for p in df.path)
if need:
    from concurrent.futures import ThreadPoolExecutor
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(build_one, list(df.itertuples())))
    print(f"cached {need} images in {time.time()-t0:.0f}s")
else:
    print("cache already built")



cached 10015 images in 389s


In [8]:

# %% ============================================ CELL 4: datasets + transforms
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0), ratio=(0.85, 1.18)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(30)], p=0.5),
    T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.03),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.25),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize(CACHE_SHORT), T.CenterCrop(IMG_SIZE),
    T.ToTensor(), T.Normalize(MEAN, STD),
])

class HAM(Dataset):
    def __init__(self, frame, tf):
        self.f = frame.reset_index(drop=True); self.tf = tf
    def __len__(self):
        return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]
        return self.tf(Image.open(r.path).convert("RGB")), int(r.y)

tr, va, te = (df[df.split == s] for s in ("train", "val", "test"))
dl_tr = DataLoader(HAM(tr, train_tf), BATCH, shuffle=True, num_workers=2,
                   pin_memory=True, drop_last=True, persistent_workers=True)
dl_va = DataLoader(HAM(va, eval_tf), 32, num_workers=2, pin_memory=True)
dl_te = DataLoader(HAM(te, eval_tf), 32, num_workers=2, pin_memory=True)


In [11]:
# %% ==================================================== CELL 5: model + loss
from transformers import RTDetrForObjectDetection

class RTDetrClassifier(nn.Module):
    """RT-DETR backbone + hybrid encoder, decoder dropped, pooled -> linear."""
    def __init__(self, name, n_classes, drop=0.2):
        super().__init__()
        det = RTDetrForObjectDetection.from_pretrained(name)   # keys match this wrapper
        base = det.model                                        # inner RTDetrModel
        self.backbone = base.backbone            # ResNet-50-vd, multi-scale out
        self.input_proj = base.encoder_input_proj
        self.encoder = base.encoder              # AIFI + CCFM
        d, nlev = base.config.d_model, len(base.encoder_input_proj)
        self.head = nn.Sequential(nn.LayerNorm(d * nlev), nn.Dropout(drop),
                                  nn.Linear(d * nlev, n_classes))
        del det, base

    def forward(self, x):
        mask = torch.ones(x.shape[0], x.shape[2], x.shape[3], device=x.device)
        feats = self.backbone(x, mask)           # [(fmap, mask), ...] 3 levels
        srcs = [self.input_proj[i](f) for i, (f, _) in enumerate(feats)]
        enc = self.encoder(inputs_embeds=srcs)[0]  # list of [B,256,h,w]
        pooled = torch.cat([f.mean((2, 3)) for f in enc], 1)  # [B, 768]
        return self.head(pooled)

model = RTDetrClassifier(MODEL_NAME, len(CLASSES)).to(DEVICE)

# sanity: shapes + parameter count
with torch.no_grad():
    model.eval()
    out = model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
print("logits:", tuple(out.shape),
      "| params:", f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M")

counts = np.bincount(tr.y.values, minlength=len(CLASSES)).astype(np.float32)
w = 1.0 / np.sqrt(counts)
w = np.clip(w / w.min(), 1.0, WEIGHT_CAP)
print("class weights:", dict(zip(CLASSES, np.round(w, 2))))
lossfn = nn.CrossEntropyLoss(weight=torch.tensor(w).to(DEVICE), label_smoothing=0.05)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
from sklearn.metrics import f1_score

@torch.no_grad()
def evaluate(loader, return_logits=False):
    model.eval()
    L, Y = [], []
    for x, y in loader:
        with torch.autocast("cuda", torch.float16, enabled=DEVICE == "cuda"):
            L.append(model(x.to(DEVICE)).float().cpu())
        Y.append(y)
    L, Y = torch.cat(L), torch.cat(Y)
    f1 = f1_score(Y.numpy(), L.argmax(1).numpy(), average="macro")
    return (f1, L, Y) if return_logits else f1

def run_epochs(n, opt, sched, tag):
    global best_f1, best_state
    for ep in range(n):
        model.train(); t0 = time.time()
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", torch.float16, enabled=DEVICE == "cuda"):
                loss = lossfn(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        sched.step()
        f1 = evaluate(dl_va)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, f"{OUT}/layer2_rtdetr_ckpt_tmp.pt")  # Colab dies
        print(f"{tag} ep {ep:2d}  val macro-F1 {f1:.4f}  ({time.time()-t0:.0f}s)")

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

logits: (2, 7) | params: 35.4M
class weights: {'akiec': np.float32(2.0), 'bcc': np.float32(2.0), 'bkl': np.float32(2.0), 'df': np.float32(2.0), 'mel': np.float32(2.0), 'nv': np.float32(1.0), 'vasc': np.float32(2.0)}


In [12]:

# %% ============================================== CELL 6: two-stage fine-tune
best_f1, best_state = -1, None

# Stage 1: head only
head_ids = {id(p) for p in model.head.parameters()}
for p in model.parameters():
    p.requires_grad_(id(p) in head_ids)
opt = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-4)
run_epochs(3, opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, 3), "S1")

# Stage 2: unfreeze all. CNN backbone can take a slightly higher LR than the
# transformer encoder (same logic as MobileNet 1e-4 vs ViT 1e-5).
for p in model.parameters():
    p.requires_grad_(True)
EPOCHS2 = 20
opt = torch.optim.AdamW([
    {"params": model.backbone.parameters(),   "lr": 5e-5},
    {"params": list(model.input_proj.parameters()) + list(model.encoder.parameters()), "lr": 2e-5},
    {"params": model.head.parameters(),       "lr": 5e-4},
], weight_decay=1e-4)
run_epochs(EPOCHS2, opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS2), "S2")

model.load_state_dict(best_state)
print(f"\nbest val macro-F1 {best_f1:.4f}")


S1 ep  0  val macro-F1 0.2569  (355s)
S1 ep  1  val macro-F1 0.2590  (347s)
S1 ep  2  val macro-F1 0.3129  (363s)
S2 ep  0  val macro-F1 0.3065  (394s)
S2 ep  1  val macro-F1 0.2704  (392s)
S2 ep  2  val macro-F1 0.4679  (392s)
S2 ep  3  val macro-F1 0.5178  (399s)
S2 ep  4  val macro-F1 0.5372  (396s)
S2 ep  5  val macro-F1 0.5298  (396s)
S2 ep  6  val macro-F1 0.6363  (393s)
S2 ep  7  val macro-F1 0.6407  (401s)
S2 ep  8  val macro-F1 0.6627  (403s)
S2 ep  9  val macro-F1 0.5700  (400s)
S2 ep 10  val macro-F1 0.6658  (401s)
S2 ep 11  val macro-F1 0.6582  (396s)
S2 ep 12  val macro-F1 0.6800  (391s)
S2 ep 13  val macro-F1 0.6992  (390s)
S2 ep 14  val macro-F1 0.6981  (392s)
S2 ep 15  val macro-F1 0.6929  (388s)
S2 ep 16  val macro-F1 0.6687  (392s)
S2 ep 17  val macro-F1 0.7083  (391s)
S2 ep 18  val macro-F1 0.6860  (398s)
S2 ep 19  val macro-F1 0.6746  (400s)

best val macro-F1 0.7083


In [13]:

# %% ============================== CELL 7: temperature scaling + test evaluation
_, lv, yv = evaluate(dl_va, return_logits=True)
lv, yv = lv.to(DEVICE), yv.to(DEVICE)
logT = torch.zeros(1, device=DEVICE, requires_grad=True)
topt = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)
def _closure():
    topt.zero_grad()
    l = F.cross_entropy(lv / logT.exp(), yv); l.backward(); return l
topt.step(_closure)
TEMP = float(logT.exp().detach())
print(f"fitted temperature T = {TEMP:.3f}")

def ece(probs, labels, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == labels).astype(np.float32)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
_, lt, yt = evaluate(dl_te, return_logits=True)
yt = yt.numpy()
p_raw = F.softmax(lt, 1).numpy()
p_cal = F.softmax(lt / TEMP, 1).numpy()
pred = p_cal.argmax(1)

MB3 = {"test_acc": accuracy_score(yt, pred),
       "test_macro_f1": f1_score(yt, pred, average="macro"),
       "ece_raw": ece(p_raw, yt), "ece_cal": ece(p_cal, yt),
       "val_macro_f1": best_f1, "temperature": TEMP}
print(f"\ntest accuracy   {MB3['test_acc']:.4f}")
print(f"test macro-F1   {MB3['test_macro_f1']:.4f}")
print(f"ECE  raw {MB3['ece_raw']:.4f}  ->  calibrated {MB3['ece_cal']:.4f}")
print("\n", classification_report(yt, pred, target_names=CLASSES, digits=3))
print(pd.DataFrame(confusion_matrix(yt, pred), index=CLASSES, columns=CLASSES))

torch.save({"state": best_state, "classes": CLASSES, "temperature": TEMP,
            "img_size": IMG_SIZE, "cache_short": CACHE_SHORT,
            "color_constancy": USE_COLOR_CONSTANCY, "weight_cap": WEIGHT_CAP,
            "model_name": MODEL_NAME, "arch": "rtdetr_encoder_cls", "metrics": MB3},
           f"{OUT}/layer2_rtdetr.pt")
json.dump({k: float(v) for k, v in MB3.items()},
          open(f"{OUT}/metrics_rtdetr.json", "w"), indent=2)
np.save(f"{OUT}/test_probs_rtdetr.npy", p_cal)   # for the ensemble in Cell 9

fitted temperature T = 0.905

test accuracy   0.8182
test macro-F1   0.6851
ECE  raw 0.0312  ->  calibrated 0.0249

               precision    recall  f1-score   support

       akiec      0.500     0.265     0.346        34
         bcc      0.845     0.798     0.821        89
         bkl      0.652     0.857     0.740       168
          df      0.714     0.556     0.625         9
         mel      0.448     0.740     0.558       146
          nv      0.974     0.846     0.905       974
        vasc      0.750     0.857     0.800        21

    accuracy                          0.818      1441
   macro avg      0.698     0.703     0.685      1441
weighted avg      0.859     0.818     0.829      1441

       akiec  bcc  bkl  df  mel   nv  vasc
akiec      9    4   12   0    8    1     0
bcc        3   71    7   0    4    1     3
bkl        2    0  144   1   15    4     2
df         0    1    2   5    0    1     0
mel        0    4   19   0  108   15     0
nv         4    4   37   1  

In [18]:
import time
x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
model.eval()
with torch.no_grad(), torch.autocast("cuda", torch.float16):
    for _ in range(5): model(x)
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(50): model(x)
    torch.cuda.synchronize()
print(f"RT-DETR enc @{IMG_SIZE}: {(time.time()-t0)/50*1000:.1f} ms/img (T4, fp16)")

RT-DETR enc @512: 36.8 ms/img (T4, fp16)


In [17]:
# %% ============ CELL 8+9 (no other weights needed)
MB1 = json.load(open(f"{OUT}/metrics_mobilenetv3.json"))
rows = ["test_acc", "test_macro_f1", "ece_raw", "ece_cal", "val_macro_f1"]
cols = {"MobileNetV3": [MB1[r] for r in rows], "RT-DETR enc": [MB3[r] for r in rows]}
vit_json = f"{OUT}/metrics_vit_small.json"
if os.path.exists(vit_json):
    MB2 = json.load(open(vit_json)); cols["ViT-S/16"] = [MB2[r] for r in rows]
print(pd.DataFrame(cols, index=rows).round(4))

routing = pd.read_csv(f"{OUT}/test_routing_detail.csv")
routing = routing.set_index("image_id").loc[te.image_id.values].reset_index()
esc = routing["escalated"].values.astype(bool)
MAL = [CLASSES.index(c) for c in ("mel", "bcc", "akiec")]
mb_pred = routing["pred"].values
if mb_pred.dtype == object:
    mb_pred = np.array([CLASSES.index(x) for x in mb_pred])

def esc_report(name, p):
    pr, yy = p[esc], yt[esc]
    mal = np.isin(yy, MAL)
    print(f"{name:14s} esc-acc {(pr==yy).mean():.4f}  esc-F1 "
          f"{f1_score(yy, pr, average='macro'):.4f}  malignant sens "
          f"{np.isin(pr[mal], MAL).mean():.3f}  benign spec {(~np.isin(pr[~mal], MAL)).mean():.3f}")

print(f"\nescalated: {esc.sum()}/{len(esc)}   bar = 0.6291")
esc_report("MobileNetV3", mb_pred)
esc_report("RT-DETR enc", pred)

               MobileNetV3  RT-DETR enc
test_acc            0.8446       0.8182
test_macro_f1       0.7059       0.6851
ece_raw             0.0289       0.0312
ece_cal             0.0249       0.0249
val_macro_f1        0.7152       0.7083

escalated: 426/1441   bar = 0.6291
MobileNetV3    esc-acc 0.6291  esc-F1 0.5468  malignant sens 0.636  benign spec 0.834
RT-DETR enc    esc-acc 0.6362  esc-F1 0.5576  malignant sens 0.769  benign spec 0.731


In [19]:
"""
OOD scores for the Decision Engine's "unknown" branch, computed on the
production Layer 1 (MobileNetV3), not on DINOv2 features.

Scores (higher = more OOD, for all three):
  msp   : 1 - max softmax               (baseline; what the gate already uses)
  energy: -T * logsumexp(logits / T)    (Liu et al. 2020)
  maha  : min_c (f-mu_c)^T Sigma^-1 (f-mu_c) on 1280-d penultimate features
          (Lee et al. 2018), class means + shared covariance fit on TRAIN.

Far-OOD set: CIFAR-10 test images pushed through the SAME preprocessing
(colour constancy, resize, crop, normalise). Reports AUROC and FPR@95%TPR,
and the threshold on val that the decision engine should use.

Needs from the ViT/RT-DETR notebook session: df, tr/va/te, HAM, MEAN/STD,
CLASSES, DEVICE, OUT, CACHE, shades_of_gray. Uses the 256px cache if it
exists; otherwise builds from source at 256.
"""

# %% ============================================= CELL O1: load production Layer 1
from torchvision.models import mobilenet_v3_large
from sklearn.metrics import roc_auc_score
import torchvision

c = torch.load(f"{OUT}/layer1_mobilenetv3_best.pt", map_location=DEVICE, weights_only=False)
mb = mobilenet_v3_large(); mb.classifier[3] = nn.Linear(1280, len(CLASSES))
mb.load_state_dict(c["state"]); mb.eval().to(DEVICE)
T_MB = c["temperature"]

# hook: penultimate = output of classifier[2] (Dropout after Hardswish), 1280-d
_feat = {}
mb.classifier[2].register_forward_hook(lambda m, i, o: _feat.__setitem__("f", o.detach()))

mb_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])

@torch.no_grad()
def feats_logits(loader):
    Fs, Ls = [], []
    for x, _ in loader:
        with torch.autocast("cuda", torch.float16, enabled=DEVICE == "cuda"):
            l = mb(x.to(DEVICE)).float()
        Fs.append(_feat["f"].float().cpu()); Ls.append(l.cpu())
    return torch.cat(Fs).numpy(), torch.cat(Ls).numpy()

# the 256px cache from the L1/ViT runs; rebuild paths if this session used 512
c256 = "/content/ham_cache"
def with_cache(frame):
    f = frame.copy(); f["path"] = f.image_id.map(lambda i: f"{c256}/{i}.jpg"); return f
if not os.path.exists(f"{c256}/{df.image_id.iloc[0]}.jpg"):
    os.makedirs(c256, exist_ok=True)
    from concurrent.futures import ThreadPoolExecutor
    def b256(row):
        if os.path.exists(row.path): return
        img = Image.open(row.src).convert("RGB"); w, h = img.size; s = 256 / min(w, h)
        img = img.resize((round(w*s), round(h*s)), Image.BICUBIC)
        img = Image.fromarray(shades_of_gray(np.array(img))); img.save(row.path, quality=95)
    with ThreadPoolExecutor(8) as ex: list(ex.map(b256, list(with_cache(df).itertuples())))

mk = lambda fr: DataLoader(HAM(with_cache(fr), mb_tf), 64, num_workers=2)
F_tr, L_tr = feats_logits(mk(tr))
F_va, L_va = feats_logits(mk(va))
F_te, L_te = feats_logits(mk(te))
print("features:", F_tr.shape, F_va.shape, F_te.shape)

# %% ============================================= CELL O2: far-OOD set (CIFAR-10)
class CifarOOD(Dataset):
    """Non-skin images through the identical preprocessing pipeline."""
    def __init__(self, n=1000):
        base = torchvision.datasets.CIFAR10("/content/cifar", train=False, download=True)
        idx = np.random.RandomState(0).choice(len(base), n, replace=False)
        self.imgs = [base[i][0] for i in idx]
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = self.imgs[i].resize((256, 256), Image.BICUBIC)
        img = Image.fromarray(shades_of_gray(np.array(img)))
        return mb_tf(img), -1

F_ood, L_ood = feats_logits(DataLoader(CifarOOD(1000), 64, num_workers=2))

# %% ============================================= CELL O3: the three scores
def sm(l): return F.softmax(torch.tensor(l) / T_MB, 1).numpy()
def msp(l):    return 1 - sm(l).max(1)
def energy(l): return (-T_MB * torch.logsumexp(torch.tensor(l) / T_MB, 1)).numpy()

# Mahalanobis: class means + shared covariance on TRAIN features
ytr = tr.y.values
mus = np.stack([F_tr[ytr == k].mean(0) for k in range(len(CLASSES))])
cov = np.cov((F_tr - mus[ytr]).T) + 1e-4 * np.eye(F_tr.shape[1])
prec = np.linalg.inv(cov)
def maha(Fx):
    d = Fx[:, None, :] - mus[None]                       # [n, C, 1280]
    return np.einsum("nci,ij,ncj->nc", d, prec, d).min(1)

SCORES = {"msp": msp, "energy": energy}
def all_scores(Fx, Lx):
    return {"msp": msp(Lx), "energy": energy(Lx), "maha": maha(Fx)}

s_va, s_te, s_ood = all_scores(F_va, L_va), all_scores(F_te, L_te), all_scores(F_ood, L_ood)

def fpr_at_95tpr(id_s, ood_s):
    thr = np.quantile(ood_s, 0.05)          # 95% of OOD above threshold
    return (id_s >= thr).mean()

print(f"{'score':8s} {'AUROC':>7s} {'FPR@95':>7s}   (ID = HAM test, OOD = CIFAR-10)")
for k in ("msp", "energy", "maha"):
    y = np.r_[np.zeros(len(s_te[k])), np.ones(len(s_ood[k]))]
    s = np.r_[s_te[k], s_ood[k]]
    print(f"{k:8s} {roc_auc_score(y, s):7.4f} {fpr_at_95tpr(s_te[k], s_ood[k]):7.3f}")

# %% ============================================= CELL O4: threshold for the engine
# Pick on VAL: flag "unknown" if score exceeds the 99th percentile of in-dist val.
# That fixes the false-unknown rate at ~1% of real skin images by construction.
BEST = "maha"      # swap to whichever won O3
unk_thr = float(np.quantile(s_va[BEST], 0.99))
print(f"\n{BEST} unknown threshold (val p99): {unk_thr:.3f}")
print(f"  test flagged unknown : {(s_te[BEST] > unk_thr).mean():.1%}   (should be ~1%)")
print(f"  CIFAR flagged unknown: {(s_ood[BEST] > unk_thr).mean():.1%}  (want >95%)")

# %% ============================================= CELL O5: do the 30 leaked misses look anomalous?
# Handoff s10 limitation 1: 30 malignant lesions are confidently called benign,
# so the softmax gate never fires. Does a feature-space score catch them?
routing = pd.read_csv(f"{OUT}/test_routing_detail.csv")
routing = routing.set_index("image_id").loc[te.image_id.values].reset_index()
leak = (routing["dangerous_miss"].values.astype(bool)) & (~routing["escalated"].values.astype(bool))
rest = ~leak
print(f"\nleaked dangerous misses: {leak.sum()}")
for k in ("energy", "maha"):
    y = np.r_[np.zeros(rest.sum()), np.ones(leak.sum())]
    s = np.r_[s_te[k][rest], s_te[k][leak]]
    p90 = np.quantile(s_va[k], 0.90)
    print(f"{k:8s} AUROC leaked-vs-rest {roc_auc_score(y, s):.3f} | "
          f"leaked above val-p90: {(s_te[k][leak] > p90).mean():.1%} vs rest {(s_te[k][rest] > p90).mean():.1%}")

np.savez(f"{OUT}/ood_scores_mobilenet.npz", mus=mus, prec=prec, unk_thr=unk_thr,
         best=BEST, **{f"te_{k}": v for k, v in s_te.items()},
         **{f"ood_{k}": v for k, v in s_ood.items()})

features: (7116, 1280) (1458, 1280) (1441, 1280)


100%|██████████| 170M/170M [41:50<00:00, 67.9kB/s]


score      AUROC  FPR@95   (ID = HAM test, OOD = CIFAR-10)
msp       0.8722   0.423
energy    0.8957   0.461
maha      0.9578   0.142

maha unknown threshold (val p99): 3600.811
  test flagged unknown : 1.3%   (should be ~1%)
  CIFAR flagged unknown: 43.1%  (want >95%)

leaked dangerous misses: 30
energy   AUROC leaked-vs-rest 0.366 | leaked above val-p90: 0.0% vs rest 8.2%
maha     AUROC leaked-vs-rest 0.520 | leaked above val-p90: 6.7% vs rest 9.6%
